# Clase 10 — NumPy aplicado y Pandas intro

En la Clase 9 conocimos los arrays de NumPy: crearlos, indexarlos y operarlos sin loops.  
Hoy vamos a profundizar un poco más en las matrices (arrays 2D) y después vamos a dar un salto natural: conocer **pandas**, la librería que le pone nombres a las columnas de nuestros datos.

**Objetivos de la clase:**
- Reorganizar arrays con `reshape` y operar por fila o por columna (`axis`).
- Entender por qué pandas existe (arrays sin nombres de columna se leen mal).
- Crear `pd.Series` y `pd.DataFrame`.
- Leer un dataset con `pd.read_csv` y explorarlo con `head()`, `info()`, `describe()`, `shape`.
- Seleccionar columnas y filas con `loc` e `iloc`.


---
## 1. Reorganizar arrays con `reshape`

`reshape` cambia la forma de un array sin cambiar sus valores. Es útil cuando los datos llegan "aplanados" (una sola fila larga) y necesitamos verlos como una tabla de filas y columnas.


In [1]:
import numpy as np

ventas_aplanadas = np.array([120, 95, 130, 110, 140, 100])

# 3 productos, 2 días de venta cada uno
ventas = ventas_aplanadas.reshape(3, 2)

print("Aplanado:", ventas_aplanadas)
print("Reorganizado (3 productos x 2 días):")
print(ventas)


Aplanado: [120  95 130 110 140 100]
Reorganizado (3 productos x 2 días):
[[120  95]
 [130 110]
 [140 100]]


---
## 2. Operar por fila o por columna con `axis`

Ya vimos `axis` en la Clase 9. Ahora lo usamos para responder preguntas concretas sobre una matriz: "¿cuánto vendió cada producto en total?" (por fila) vs. "¿cuánto se vendió cada día en total?" (por columna).

- `axis=1` → recorre las columnas y da un resultado **por fila**.
- `axis=0` → recorre las filas y da un resultado **por columna**.


In [2]:
ventas = np.array([
    [120, 95],
    [130, 110],
    [140, 100],
])

total_por_producto = ventas.sum(axis=1)   # suma las columnas de cada fila
total_por_dia = ventas.sum(axis=0)        # suma las filas de cada columna

print("Total por producto:", total_por_producto)
print("Total por día      :", total_por_dia)


Total por producto: [215 240 240]
Total por día      : [390 305]


---
## 3. ¿Por qué pandas?

En la matriz de ventas de arriba, la fila 0 es un producto... pero ¿cuál? NumPy no guarda nombres, solo números en posiciones. Para datos reales necesitamos algo que recuerde: "esta columna es `calorias`", "esta fila es la manzana roja".

**pandas** resuelve justo eso: es como un array de NumPy, pero con nombres de columna y de fila. Por abajo, de hecho, pandas usa NumPy.


---
## 4. `pd.Series`: un array con etiquetas

Una `Series` es una columna con nombre: cada valor tiene una etiqueta (el índice), en vez de solo una posición.


In [3]:
import pandas as pd

calorias = pd.Series(
    [95, 80, 62, 45],
    index=["manzana roja", "manzana verde", "naranja de mesa", "banana"],
)

print(calorias)
print()
print("Calorías de la banana:", calorias["banana"])
print("Promedio de calorías :", calorias.mean())


manzana roja       95
manzana verde      80
naranja de mesa    62
banana             45
dtype: int64

Calorías de la banana: 45
Promedio de calorías : 70.5


---
## 5. `pd.DataFrame`: varias columnas con nombre

Un `DataFrame` es una tabla: varias `Series` que comparten el mismo índice. Se puede crear desde un diccionario, donde cada clave es una columna.

Vamos a usar el mismo dataset de frutas que ya conocemos.


In [4]:
frutas = pd.DataFrame({
    "nombre": ["manzana roja", "manzana verde", "naranja de mesa", "banana", "pera"],
    "clase": ["manzana", "manzana", "naranja", "banana", "pera"],
    "color": ["rojo", "verde", "naranja", "amarillo", "verde"],
    "calorias": [95, 80, 62, 89, 57],
})

frutas


,nombre,clase,color,calorias
0,manzana roja,manzana,rojo,95
1,manzana verde,manzana,verde,80
2,naranja de mesa,naranja,naranja,62
3,banana,banana,amarillo,89
4,pera,pera,verde,57


---
## 6. Leer datos con `pd.read_csv` y explorarlos

Lo más común no es escribir el DataFrame a mano, sino leerlo desde un archivo CSV con `pd.read_csv("archivo.csv")`. Para practicar sin depender de un archivo en disco, simulamos ese CSV como texto.

Apenas cargamos un dataset, conviene explorarlo antes de tocar nada:

| Método / atributo | Qué muestra |
|---|---|
| `.head()` | Las primeras filas |
| `.info()` | Columnas, tipos de dato y nulos |
| `.describe()` | Estadísticas de las columnas numéricas |
| `.shape` | (cantidad de filas, cantidad de columnas) |


In [5]:
from io import StringIO

csv_como_texto = StringIO('''nombre,clase,color,calorias
manzana roja,manzana,rojo,95
manzana verde,manzana,verde,80
naranja de mesa,naranja,naranja,62
banana,banana,amarillo,89
pera,pera,verde,57
kiwi,kiwi,verde,61
''')

frutas_csv = pd.read_csv(csv_como_texto)

print("Forma del dataset:", frutas_csv.shape)
frutas_csv.head()


Forma del dataset: (6, 4)


,nombre,clase,color,calorias
0,manzana roja,manzana,rojo,95
1,manzana verde,manzana,verde,80
2,naranja de mesa,naranja,naranja,62
3,banana,banana,amarillo,89
4,pera,pera,verde,57


In [6]:
frutas_csv.info()
print()
frutas_csv.describe()


<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   nombre    6 non-null      str  
 1   clase     6 non-null      str  
 2   color     6 non-null      str  
 3   calorias  6 non-null      int64
dtypes: int64(1), str(3)
memory usage: 324.0 bytes



,calorias
count,6.000000
mean,74.000000
std,16.149303
min,57.000000
25%,61.250000
50%,71.000000
75%,86.750000
max,95.000000


---
## 7. Seleccionar columnas y filas: `loc` e `iloc`

- Una columna: `df["nombre_columna"]`.
- Varias columnas: `df[["col1", "col2"]]`.
- Filas por **etiqueta** de índice: `df.loc[...]`.
- Filas por **posición** numérica: `df.iloc[...]`.


In [7]:
solo_nombres = frutas_csv["nombre"]
nombre_y_calorias = frutas_csv[["nombre", "calorias"]]

primera_fila_por_posicion = frutas_csv.iloc[0]
primera_fila_por_etiqueta = frutas_csv.loc[0]

print(solo_nombres.tolist())
print()
print(nombre_y_calorias)
print()
print("iloc[0]:\n", primera_fila_por_posicion)


['manzana roja', 'manzana verde', 'naranja de mesa', 'banana', 'pera', 'kiwi']

            nombre  calorias
0     manzana roja        95
1    manzana verde        80
2  naranja de mesa        62
3           banana        89
4             pera        57
5             kiwi        61

iloc[0]:
 nombre      manzana roja
clase            manzana
color               rojo
calorias              95
Name: 0, dtype: object


---
## 📝 Actividad 1 — Ventas por producto y por día

**Consigna:**
- Reorganizar `ventas_semana` (12 valores) en una matriz de 4 productos x 3 días con `reshape`.
- Calcular el total vendido por producto (`axis=1`) y el total vendido por día (`axis=0`).


In [8]:
# ACTIVIDAD 1: ventas por producto y por día
ventas_semana = np.array([120, 95, 130, 110, 140, 100, 90, 105, 115, 125, 135, 145])

# TODO: reorganizar en 4 productos x 3 días
ventas_matriz = ventas_semana.reshape(4, 3)

# TODO: total por producto y total por día
total_por_producto = ventas_matriz.sum(axis=1)
total_por_dia = ventas_matriz.sum(axis=0)

print("Matriz de ventas:")
print(ventas_matriz)
print("Total por producto:", total_por_producto)
print("Total por día      :", total_por_dia)


Matriz de ventas:
[[120  95 130]
 [110 140 100]
 [ 90 105 115]
 [125 135 145]]
Total por producto: [345 350 310 405]
Total por día      : [445 475 490]


---
### Actividad 2 — Explorar el dataset de frutas

**Consigna:**
- A partir de `frutas_csv`, mostrar solo las columnas `nombre` y `color`.
- Usar `loc` para mostrar la fila de la fruta que está en la posición 3.
- Calcular el promedio de `calorias` de todo el dataset.


In [9]:
# ACTIVIDAD 2: explorar el dataset de frutas
# TODO: mostrar solo nombre y color
nombre_y_color = frutas_csv[["nombre", "color"]]

# TODO: mostrar la fila en la posición de índice 3
fila_indice_3 = frutas_csv.loc[3]

# TODO: promedio de calorías
promedio_calorias = frutas_csv["calorias"].mean()

print(nombre_y_color)
print()
print("Fila en índice 3:\n", fila_indice_3)
print()
print("Promedio de calorías:", promedio_calorias)


            nombre     color
0     manzana roja      rojo
1    manzana verde     verde
2  naranja de mesa   naranja
3           banana  amarillo
4             pera     verde
5             kiwi     verde

Fila en índice 3:
 nombre        banana
clase         banana
color       amarillo
calorias          89
Name: 3, dtype: object

Promedio de calorías: 74.0


---
## ✅ Resumen

| Herramienta | Qué resuelve |
|---|---|
| `array.reshape(...)` | Reorganizar valores sin cambiarlos |
| `array.sum(axis=1)` vs `axis=0` | Operar por fila o por columna |
| `pd.Series(...)` | Array con etiquetas |
| `pd.DataFrame(...)` | Tabla con columnas con nombre |
| `pd.read_csv(...)` | Cargar datos tabulares |
| `.head()`, `.info()`, `.describe()`, `.shape` | Explorar un dataset antes de tocarlo |
| `df[col]`, `df.loc[...]`, `df.iloc[...]` | Seleccionar columnas y filas |

**Lo que construiste hoy:**
- matrices reorganizadas y resumidas por fila/columna,
- tu primer `DataFrame` de pandas, cargado y explorado.

**Lo que viene:**
- **Clase 11**: vamos a manipular el dataset de frutas en profundidad: filtrar, limpiar valores faltantes, agrupar y combinar datos.
